# 02. district_population.csv 전처리

## 개요

서울시 25개 자치구 × 연도(2020~2024) 단위의 **등록인구** 및 **종사자수** 데이터셋을 생성합니다.  
이 파일은 `waste_by_district_year.csv`의 `household_waste`, `biz_nonbiz_waste` 파생 변수 계산에도 사용됩니다.

### 사용 원본 파일

| 파일명 | 추출 변수 |
|--------|----------|
| `등록인구_연령별_동별__20260430162035.csv` | population (D열 = 합계) |
| `사업체현황_종사자규모별_동별__20260430163439.csv` | workers (E열 = 종사자수) |

### 열 위치 메모

- **등록인구**: col0=동별(자치구), col1=항목(한국인), col2=시점(연도), col3=합계(D열)
- **종사자**: col0=동별(1), col1=동별(2, 자치구), col2=시점(연도), col4=종사자수(E열)

### 출력 파일
- `district_population.csv`

## 0. 라이브러리 및 경로 설정

In [ ]:
import pandas as pd
import numpy as np
import os

BASE_DIR = os.path.dirname(os.path.abspath('__file__'))
DATA_DIR = os.path.join(BASE_DIR, 'data')
OUT_DIR  = os.path.join(BASE_DIR, 'output')
os.makedirs(OUT_DIR, exist_ok=True)

GU_LIST = [
    '종로구','중구','용산구','성동구','광진구','동대문구','중랑구',
    '성북구','강북구','도봉구','노원구','은평구','서대문구','마포구',
    '양천구','강서구','구로구','금천구','영등포구','동작구','관악구',
    '서초구','강남구','송파구','강동구'
]
YEARS = [2020, 2021, 2022, 2023, 2024]

print(f'DATA_DIR : {DATA_DIR}')
print(f'OUT_DIR  : {OUT_DIR}')


## 1. 등록인구 데이터 로드

### 구조
- col0: 자치구명 (동별1)
- col1: 항목 (한국인 고정)
- col2: 연도
- col3: 합계 인구 **(D열)** → `population`

### 필터 조건
- col0이 25개 자치구명 중 하나인 행만 추출
- col1이 '한국인'인 행 (외국인 포함 합산 행 제외)

In [ ]:
# 등록인구: col0=자치구, col1='한국인'(고정), col2=연도, col3=합계인구(D열)
df_raw_pop = pd.read_csv(
    os.path.join(DATA_DIR, '등록인구_연령별_동별__20260430162035.csv'),
    encoding='utf-8-sig', header=None
)

df_pop = df_raw_pop[df_raw_pop[0].isin(GU_LIST)].copy()
df_pop = df_pop[[0, 2, 3]].copy()
df_pop.columns = ['district', 'year', 'population']

df_pop['population'] = pd.to_numeric(
    df_pop['population'].astype(str).str.replace(',', ''), errors='coerce'
).fillna(0).astype(int)
df_pop['year'] = df_pop['year'].astype(int)
df_pop = df_pop[df_pop['year'].isin(YEARS)].reset_index(drop=True)

print(f'shape: {df_pop.shape}  (기대: {len(GU_LIST)*len(YEARS)} 행)')
df_pop.head(10)


## 2. 종사자수 데이터 로드

### 구조
- col0: 동별(1) — '합계' 고정
- col1: 동별(2) — 자치구명
- col2: 연도
- col3: 사업체수 (사용 안 함)
- col4: 종사자수 **(E열)** → `workers`

### 필터 조건
- col1이 25개 자치구명 중 하나인 행만 추출

In [ ]:
# 종사자: col0='합계'(고정), col1=자치구, col2=연도, col4=종사자수(E열)
df_raw_wk = pd.read_csv(
    os.path.join(DATA_DIR, '사업체현황_종사자규모별_동별__20260430163439.csv'),
    encoding='utf-8-sig', header=None
)

df_wk = df_raw_wk[df_raw_wk[1].isin(GU_LIST)].copy()
df_wk = df_wk[[1, 2, 4]].copy()
df_wk.columns = ['district', 'year', 'workers']

df_wk['workers'] = pd.to_numeric(
    df_wk['workers'].astype(str).str.replace(',', ''), errors='coerce'
).fillna(0).astype(int)
df_wk['year'] = df_wk['year'].astype(int)
df_wk = df_wk[df_wk['year'].isin(YEARS)].reset_index(drop=True)

print(f'shape: {df_wk.shape}  (기대: {len(GU_LIST)*len(YEARS)} 행)')
df_wk.head(10)


## 3. 등록인구 + 종사자수 병합

district + year 기준으로 inner join합니다.  
두 파일 모두 25개 자치구 × 5개 연도 구조이므로 125행이 기대됩니다.

In [ ]:
df_result = df_pop.merge(df_wk, on=['district', 'year'], how='inner')

df_result['district'] = pd.Categorical(df_result['district'], categories=GU_LIST, ordered=True)
df_result = df_result.sort_values(['district', 'year']).reset_index(drop=True)

print(f'병합 결과 shape: {df_result.shape}  (기대: {len(GU_LIST)*len(YEARS)} × 4)')
print(f'결측값 합계    : {df_result.isnull().sum().sum()}')
df_result.head(10)


## 4. 검증

자치구별 연도별 인구 및 종사자수 요약을 출력해 이상값을 확인합니다.

In [ ]:
print('=== 자치구별 2024년 인구 · 종사자 요약 ===')
summary = df_result[df_result['year']==2024][['district','population','workers']].copy()
summary['worker_ratio'] = (summary['workers'] / summary['population']).round(2)
print(summary.to_string(index=False))

print()
print('=== 전체 기술통계 ===')
print(df_result[['population','workers']].describe().round(0))


## 5. 출력 저장

In [ ]:
out_path = os.path.join(OUT_DIR, 'district_population.csv')
df_result.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'✓ 저장 완료: {out_path}')
print(f'  파일 크기: {os.path.getsize(out_path):,} bytes')
